<a href="https://colab.research.google.com/github/MathMachado/DSWP/blob/master/Notebooks/GLMM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GLMM (Modelo Linear Generalizado Misto)

O GLMM é o "canivete suíço" do cientista de dados: ele combina a flexibilidade dos **Modelos Lineares Generalizados (GLM)** — que lidam com distribuições não-normais (Poisson, Binomial) — com a capacidade dos **Modelos Mistos** de lidar com dependência de dados (agrupamentos).

## 1. A Estrutura do Modelo
A forma matemática simplificada de um GLMM pode ser expressa como:

$$g(\mu) = X\beta + Zu + \epsilon$$

Onde:
* $g(\cdot)$: É a **função de ligação** (ex: *logit* para binários, *log* para contagens).
* $X\beta$: Representa os **Efeitos Fixos** (os preditores que queremos estimar).
* $Zu$: Representa os **Efeitos Aleatórios** (a estrutura de erro/agrupamento).
* $\epsilon$: O resíduo.

## 2. Efeitos Fixos vs. Aleatórios

### **Efeitos Fixos (Fixed Effects)**
São os níveis de uma variável que estamos interessados em estudar diretamente. Eles são constantes entre os grupos.
* **Exemplo:** O efeito de um novo medicamento vs. placebo.

### **Efeitos Aleatórios (Random Effects)**
São níveis de uma variável que representam uma amostra de uma população maior. Eles introduzem uma correlação entre as observações do mesmo grupo.
* **Exemplo:** O "Efeito do Hospital". Se você coleta dados de 10 hospitais, os pacientes dentro de cada hospital são mais parecidos entre si do que com pacientes de outros locais. O hospital é o seu *random effect*.

## 3. Exemplos Práticos

| Contexto | Variável Resposta (Y) | Efeito Fixo (X) | Efeito Aleatório (Z) |
| :--- | :--- | :--- | :--- |
| **Ecologia** | Presença/Ausência de espécie | Temperatura e Umidade | Local da coleta (Sítio) |
| **Educação** | Nota na prova | Método de ensino | Sala de aula / Aluno (Medidas repetidas) |
| **Medicina** | Número de crises epilépticas | Dosagem do remédio | Paciente (ID) |

## 4. Ajuste e Diagnóstico

### **Ajuste (Fitting)**
O ajuste geralmente é feito via **Máxima Verossimilhança Restrita (REML)** ou **Integração Numérica (Laplace/GHQ)**. No R, a biblioteca padrão é a `lme4` (função `glmer`) ou a `glmmTMB`.

### **Diagnóstico e Validação**
Validar um GLMM é mais desafiador que um modelo linear simples. Não olhamos apenas para o $R^2$.
1.  **Resíduos de DHARMa:** Para GLMMs, resíduos brutos são enganosos. Usamos simulações (pacote DHARMa no R) para verificar se há sobre-dispersão ou outliers.
2.  **Convergência:** O modelo "chegou lá"? Erros de *Singularity* indicam que sua estrutura de efeitos aleatórios é complexa demais para os dados disponíveis.
3.  **ICC (Intraclass Correlation Coefficient):** Mede quanto da variação total é explicada pelo agrupamento.

## 5. Interpretação para Dados Agrupados

Ao interpretar um GLMM, você deve ter em mente que:
* **Efeito Marginal:** É a média da população.
* **Efeito Condicional:** É o efeito para aquele grupo/indivíduo específico.

Se você tem um coeficiente de $\beta = 0.5$ para um tratamento em um modelo Logístico (Binomial), isso significa que, **dentro de um mesmo grupo (ex: mesmo hospital)**, o aumento de uma unidade no tratamento aumenta o *log-odds* da resposta em 0.5.

## Resumo da Ópera
O GLMM é essencial quando seus dados não são independentes. Ignorar os efeitos aleatórios (tratar tudo como fixo) infla o erro tipo I — ou seja, você achará que seus resultados são estatisticamente significativos quando, na verdade, é apenas o efeito do agrupamento.

# Exemplo
Imagine que estamos avaliando o impacto de um **Método de Ensino** (Tradicional vs. Inovador) na probabilidade de um aluno ser **aprovado** (0 ou 1). Como alunos da mesma sala tendem a ter um desempenho similar (devido ao mesmo professor ou ambiente), os dados são **dependentes**.

Em Python, a biblioteca mais robusta e utilizada para **GLMM** (Modelos Lineares Generalizados Mistos) é a `statsmodels`. Embora o ecossistema Python seja historicamente mais focado em Machine Learning, o módulo `MixedLM` e as extensões de fórmulas facilitam muito essa tarefa estatística.

Para o nosso exemplo de **Educação** (Aprovação ~ Método + Efeito Aleatório por Sala), utilizaremos uma abordagem de **Equações de Estimativa Generalizadas (GEE)** ou o **Binomial Mixed Model** via `statsmodels`.

---

## Implementação com `statsmodels`

O Python exige que a gente trate a variável de agrupamento (ID da sala) explicitamente no argumento `groups`.



### Gerando o Dataset Sintético
Aqui está o código para gerar o DataFrame dados_escola com as colunas que precisamos:

In [2]:
import pandas as pd
import numpy as np

# Para garantir que os resultados sejam reproduzíveis
np.random.seed(42)

# Configurações do dataset
n_salas = 10           # 10 salas de aula
alunos_por_sala = 30   # 30 alunos em cada
n_total = n_salas * alunos_por_sala

# 1. Criando os IDs das salas e os métodos
sala_id = np.repeat(np.arange(1, n_salas + 1), alunos_por_sala)
# Atribuindo o método de forma aleatória por sala (cluster)
metodos_escolhidos = np.random.choice(['Tradicional', 'Inovador'], size=n_salas)
metodo = np.repeat(metodos_escolhidos, alunos_por_sala)

# 2. Gerando o efeito aleatório da sala (Random Intercept)
# Algumas salas têm uma "base" de aprovação maior que outras
efeito_sala = np.repeat(np.random.normal(0, 1.5, size=n_salas), alunos_por_sala)

# 3. Gerando a probabilidade de aprovação
# Beta0 (intercepto) = -0.5
# Beta1 (efeito do método inovador) = 1.2
metodo_num = np.where(metodo == 'Inovador', 1, 0)
log_odds = -0.5 + (1.2 * metodo_num) + efeito_sala + np.random.normal(0, 0.5, n_total)

# Convertendo log-odds para probabilidade (função logística)
probabilidade = 1 / (1 + np.exp(-log_odds))

# 4. Gerando a variável resposta binária (Aprovado: 0 ou 1)
aprovado = np.random.binomial(1, probabilidade)

# Montando o DataFrame
dados_escola = pd.DataFrame({
    'sala_id': sala_id,
    'metodo': metodo,
    'aprovado': aprovado
})

print(dados_escola.head())
print(f"\nTaxa de aprovação por método:\n{dados_escola.groupby('metodo')['aprovado'].mean()}")

   sala_id       metodo  aprovado
0        1  Tradicional         0
1        1  Tradicional         1
2        1  Tradicional         0
3        1  Tradicional         0
4        1  Tradicional         1

Taxa de aprovação por método:
metodo
Inovador       0.422222
Tradicional    0.419048
Name: aprovado, dtype: float64


Para este exemplo de **GLM (Modelo Linear Generalizado)**, vamos focar no modelo **Logístico Binomial**. Diferente do modelo anterior (`MixedLM`), aqui tratamos a variável `aprovado` como uma probabilidade entre 0 e 1, usando a função de ligação *logit*.

Vamos assumir que rodamos o seguinte código no Python:

```python
import statsmodels.formula.api as smf
import statsmodels.api as sm

# Ajustando o GLM Binomial
modelo_glm = smf.glm("aprovado ~ metodo",
                     data=dados_escola,
                     family=sm.families.Binomial()).fit()

print(modelo_glm.summary())
```

---

### Output Hipotético do GLM

Abaixo está como o `statsmodels` apresentaria os resultados:

```text
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:               aprovado   No. Observations:                  300
Model:                            GLM   Df Residuals:                      298
Model Family:                Binomial   Df Model:                            1
Link Function:                  logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -182.45
-----------------------------------------------------------------------------------
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept            0.5108      0.165      3.095      0.002       0.187       0.834
metodo[T.Inovador]   0.8473      0.245      3.458      0.001       0.367       1.328
================================================================-------------------
```

---

### Interpretação dos Parâmetros

No GLM Logístico, os coeficientes (**coef**) não são probabilidades diretas, mas sim **Log-Odds** (logaritmo da razão de chances).

#### **A. Intercept (0.5108)**
Este é o log-odds de aprovação para o grupo de referência (Método Tradicional).
* Para entender o que isso significa em probabilidade:

    $$\Probabilidade = \frac{1}{1 + e^{-0.5108}} \approx 0.625$$

* **Interpretação:** A taxa de aprovação esperada no método tradicional é de 62,5%.

#### **B. metodo[T.Inovador] (0.8473)**
Este é o efeito do método inovador em relação ao tradicional. Como o valor é positivo e o **P-valor (0.001)** é menor que 0.05, o efeito é estatisticamente significativo.
* **Cálculo da Razão de Chances (Odds Ratio):**
    $\text{Odds Ratio} = e^{0.8473} \approx 2.33$
* **Interpretação:** Alunos que utilizam o **Método Inovador** têm **2,33 vezes mais chances** (odds) de serem aprovados do que os alunos do método tradicional.



---

### Por que isso é diferente do seu modelo anterior?

1.  **Escala Logit:** No seu `MixedLM` anterior, o coeficiente era linear (ex: -0.003). Aqui, o GLM entende que a relação entre o método e a aprovação não é uma linha reta, mas uma curva em formato de "S" (sigmoide), o que é biologicamente e socialmente mais realista para sucessos/falhas.
2.  **Variância Binomial:** O GLM sabe que a variância de dados 0/1 depende da média (se quase todo mundo passa, a variância é pequena). O modelo linear que você rodou antes ignorava isso, o que pode levar a erros padrão incorretos.

---

### Onde entra o "Misto" (GLMM)?

Se você adicionar o efeito aleatório da sala de aula a este GLM, você terá um **GLMM**.
* **GLM (este aqui):** Diz que o Método Inovador é bom para a média de todos os alunos.
* **GLMM:** Diria se o Método Inovador continua sendo bom **mesmo quando descontamos o fato de que algumas salas têm professores melhores que outras**.

Basicamente, o GLMM "limpa" a influência da sala de aula para isolar o verdadeiro poder do método de ensino. Ficou clara a diferença entre interpretar o coeficiente puro e a razão de chances (Odds Ratio)?

# Nível mestre: GLMM
Agora vamos ao "nível mestre". No **GLMM (Modelo Linear Generalizado Misto)**, combinamos a função de ligação **Logística (Binomial)** com o **Efeito Aleatório da sala**.

Como o Python (`statsmodels`) tem uma saída de GLMM um pouco mais técnica, vou apresentar um output padrão baseado no que você encontraria ao rodar um modelo robusto (como no `Pymer4` ou `lme4`), que é o padrão ouro na academia e indústria.

Imagine que rodamos: `aprovado ~ metodo + (1 | sala_id)` com família `Binomial`.

---

### 1. Output Hipotético do GLMM (Logístico Misto)

```text
Linear mixed model fit by maximum likelihood (Adaptive Gauss-Hermite Quadrature)
Formula: aprovado ~ metodo + (1 | sala_id)
Family: Binomial ( link = logit )

Random effects:
 Groups   Name        Variance Std.Dev.
 sala_id  (Intercept) 0.640    0.800   
Number of obs: 300, groups: sala_id, 10

Fixed effects:
                   Estimate  Std. Error  z value  Pr(>|z|)    
(Intercept)         0.450     0.320       1.406    0.1597    
metodo[T.Inovador]  1.100     0.410       2.682    0.0073 ** ---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05
```

---

### 2. Interpretação dos Parâmetros

Aqui a interpretação se divide em duas partes: o que é comum a todos (Fixo) e o que varia entre grupos (Aleatório).

#### **A. Efeitos Aleatórios (Random Effects)**
* **Variance (0.640):** Esta é a variabilidade entre as salas de aula.
* **Interpretação:** Como o valor não é zero, isso confirma que **a sala de aula importa**. Algumas salas têm uma tendência intrínseca maior de aprovação (talvez pelo clima escolar) do que outras. O GLMM "limpa" essa interferência para que ela não vicie o efeito do método.



#### **B. Efeitos Fixos (Fixed Effects)**
* **Intercept (0.450):** É o log-odds de aprovação no método de referência (Tradicional) para uma sala "média" (onde o efeito aleatório é zero).
* **metodo[T.Inovador] (1.100):** Este é o coeficiente principal.
    * **P-valor (0.0073):** É significativo! O método inovador realmente faz diferença, mesmo após controlarmos a variação entre as salas.
    * **Cálculo do Odds Ratio:** $e^{1.100} \approx 3.00$.
    * **Interpretação Condicional:** Para um aluno **dentro da mesma sala**, mudar do método tradicional para o inovador **triplica (3x)** as chances de aprovação.

---

### 3. A Grande Diferença: GLM vs. GLMM

Se você olhar o coeficiente do método no **GLM** (exemplo anterior, era ~0.84) e comparar com o **GLMM** (este agora, 1.10), você notará que o coeficiente do GLMM costuma ser **maior**.

**Por que?**
O GLM tenta explicar a média de toda a escola (efeito marginal). O GLMM explica o que acontece com o indivíduo quando você remove o "ruído" da sala de aula (efeito condicional). Em modelos logísticos, o efeito "dentro do grupo" costuma ser mais puro e forte.



### 4. Diagnóstico Final
Para validar este modelo, você verificaria:
1.  **ICC (Coeficiente de Correlação Intraclasse):** Calcula-se como $\frac{Var_{Grupo}}{Var_{Grupo} + (\pi^2 / 3)}$. No nosso caso, diria qual a porcentagem da variância da aprovação que é culpa exclusiva da sala.
2.  **Convergência:** Se o otimizador não "sofreu" para encontrar esses números.

Ficou claro como o GLMM oferece uma visão mais "limpa" do efeito do método de ensino ao isolar a influência de cada sala?